# 02 Training MLflow Tracking and Model Registry

**Purpose:** Train a model using the enriched training set, log experiments to MLflow, register the model in the Model Registry, add a signature, and simulate a CI promotion to Staging with a smoke test.

**Run order:** Execute cells sequentially on a Databricks cluster. This notebook uses the `demo.enriched_training_set` Delta table produced by `01-Feature-Engineering-and-Point-in-Time.ipynb`.

## 1 Install / verify dependencies
Run this cell if the cluster session does not already have the required packages installed.

In [ ]:
%pip install -q mlflow scikit-learn pandas joblib

import importlib
for pkg in ["mlflow","sklearn","pandas","joblib"]:
    try:
        m = importlib.import_module(pkg)
        print(pkg, getattr(m, "__version__", "unknown"))
    except Exception as e:
        print(pkg, "not available:", e)

## 2 Load enriched training set and prepare features/labels
This cell reads `demo.enriched_training_set` and prepares a simple train/test split. Adjust feature selection for your use case.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd
from sklearn.model_selection import train_test_split

spark = SparkSession.builder.getOrCreate()

# Read enriched training set
df = spark.table("demo.enriched_training_set").toPandas()
print("Loaded rows:", len(df))

# Basic preprocessing: drop rows with null label or features
df = df.dropna(subset=["label"])

# Select features (example): use f1, f2, avg_f1, avg_f2, num_events
feature_cols = [c for c in ["f1","f2","avg_f1","avg_f2","num_events"] if c in df.columns]
X = df[feature_cols].fillna(0.0)
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train rows:", len(X_train), "Test rows:", len(X_test))

## 3 Train a model and log to MLflow
We train a `RandomForestClassifier`, log parameters, metrics, and the model artifact to MLflow. The run id is captured for later registration.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
import numpy as np

mlflow.set_experiment("demo_training_experiment")

with mlflow.start_run() as run:
    run_id = run.info.run_id
    params = {"model_type": "RandomForest", "n_estimators": 100, "random_state": 42}
    mlflow.log_params(params)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)

    mlflow.log_metric("accuracy", float(acc))
    mlflow.log_metric("precision", float(prec))
    mlflow.log_metric("recall", float(rec))

    # Log the model artifact
    mlflow.sklearn.log_model(model, artifact_path="model")

print("Run completed. run_id:", run_id)
print("Accuracy:", acc, "Precision:", prec, "Recall:", rec)

## 4 Register the model in MLflow Model Registry
Create a registered model (if not exists) and create a model version from the run artifact. Add a description and a tag with the accuracy metric.

In [ ]:
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature
import pandas as pd

client = MlflowClient()
registered_model_name = "demo_rf_model"

# Create registered model if it doesn't exist
try:
    client.create_registered_model(registered_model_name)
    print(f"Created registered model: {registered_model_name}")
except Exception as e:
    # model may already exist
    print(f"Registered model exists or creation failed: {e}")

# Build model URI from the run
model_uri = f"runs:/{run_id}/model"
mv = client.create_model_version(name=registered_model_name, source=model_uri, run_id=run_id)
print("Created model version:", mv.version)

# Infer signature from a small sample
sample_input = X_train.head(5)
signature = infer_signature(sample_input, model.predict(sample_input))
client.update_model_version(name=registered_model_name, version=mv.version, description="Initial version with signature")
client.set_model_version_tag(registered_model_name, mv.version, "accuracy", str(acc))
print("Updated model version with signature and tags")

## 5 Add model signature and save a model with signature (optional)
Demonstrate saving a model with an explicit signature and sample input as an artifact (useful for model validation and serving).

In [ ]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec
import mlflow.sklearn

# Build a simple schema for the features
input_schema = Schema([ColSpec("double", name=c) for c in feature_cols])
output_schema = Schema([ColSpec("integer")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Log a second model artifact with signature (optional)
with mlflow.start_run() as run2:
    mlflow.log_param("model_type", "RandomForest-with-signature")
    mlflow.sklearn.log_model(model, artifact_path="model_with_signature", signature=signature, input_example=X_train.head(3))
    print("Logged model_with_signature in run:", run2.info.run_id)

## 6 Simulate CI promotion: transition model version to Staging and run a smoke test
This cell transitions the created model version to `Staging` and performs a quick smoke test by loading the model from the registry and running a few predictions.

In [ ]:
# Transition to Staging
try:
    client.transition_model_version_stage(name=registered_model_name, version=mv.version, stage="Staging", archive_existing_versions=False)
    print(f"Transitioned model {registered_model_name} version {mv.version} to Staging")
except Exception as e:
    print("Stage transition failed or already in desired stage:", e)

# Smoke test: load model from registry Staging and run predictions
import mlflow.pyfunc
model_uri_staging = f"models:/{registered_model_name}/Staging"
try:
    loaded = mlflow.pyfunc.load_model(model_uri_staging)
    sample = X_test.head(3)
    preds = loaded.predict(sample)
    print("Smoke test predictions:", preds.tolist())
    assert len(preds) == 3, "Unexpected prediction length in smoke test"
    print("Smoke test passed")
except Exception as e:
    print("Smoke test failed:", e)


## 7 Tests and assertions
Run a few checks to validate the registry entry, tags, and that the model in Staging returns predictions.

In [ ]:
# Test 1: Registered model exists
models = [m.name for m in client.list_registered_models()]
assert registered_model_name in models, f"Registered model {registered_model_name} not found"
print("Registered model present")
# Test 2: Model version in Staging
versions = client.get_latest_versions(registered_model_name)
staging_versions = [v for v in versions if v.current_stage == "Staging"]
assert len(staging_versions) > 0, "No model version in Staging"
print("Model version in Staging confirmed")
# Test 3: Tag accuracy exists
tags = client.get_model_version(registered_model_name, mv.version).tags
assert "accuracy" in tags, "Accuracy tag missing on model version"
print("Accuracy tag present with value:", tags.get("accuracy"))

## 8 Notes and next steps
- Replace simulated promotion with your CI system calling the Model Registry API to transition stages after automated checks.  
- Add more robust validation tests (data schema checks, performance thresholds, adversarial tests) before promoting to Production.  
- Consider adding model metadata (training data snapshot, feature lineage, hyperparameters) to the registry for traceability.  
- Next notebook: `03-Serving-and-Monitoring.ipynb` to deploy and validate serving behavior.